# 🔬 SoccerNet x StatsBomb — Sync Validation
## BBC vs MSN — Tactical Action Classifier

Validates whether SoccerNet video timestamps align with StatsBomb event timestamps.

**Test match:** El Clasico — Real Madrid 0-4 Barcelona (La Liga 2015/16)
**Strategy:** Extract frames at goal timestamps from both sources and compare visually.

---

| Section | Description |
|---|---|
| **1** | Setup — mount Drive, install deps, init variables |
| **2** | Download Labels-v2.json from SoccerNet |
| **3** | Compare SoccerNet vs StatsBomb timestamps |
| **4** | Extract frames at goal timestamps |
| **5** | Visual validation + offset calculation |
| **6** | Sync verdict |

> ⚠️ Run sections **in order**. Each section depends on the previous one.

---
# ⚙️ Section 1 — Setup
Run this **first** every time you open the notebook.

In [1]:
# ============================================================
# SECTION 1 — Setup
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import subprocess
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ------------------------------------------------------------
# Install dependencies
# ------------------------------------------------------------
print('Installing dependencies...')
for pkg in ['SoccerNet', 'pyocclient', 'statsbombpy']:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True
    )
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'   {pkg}: {status}')

from SoccerNet.Downloader import SoccerNetDownloader
from statsbombpy import sb

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT_ROOT = '/content/drive/MyDrive/bbc-vs-msn-classifier'
VIDEO_DIR    = os.path.join(PROJECT_ROOT, 'data/raw')
FRAMES_DIR   = os.path.join(PROJECT_ROOT, 'data/frames/validation')
os.makedirs(FRAMES_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# ------------------------------------------------------------
# SoccerNet credentials
# ------------------------------------------------------------
key_path = os.path.join(PROJECT_ROOT, 'soccernet_key.txt')
with open(key_path, 'r') as f:
    SN_PASSWORD = f.read().strip()

mySNdl          = SoccerNetDownloader(LocalDirectory=VIDEO_DIR)
mySNdl.password = SN_PASSWORD

# ------------------------------------------------------------
# Test match — El Clasico (4 goals, easy to identify visually)
# ------------------------------------------------------------
MATCH_GAME  = '2015-11-21 - 20-15 Real Madrid 0 - 4 Barcelona'
COMP_KEY    = 'laliga'
LEAGUE_DIR  = 'spain_laliga'
MATCH_PATH  = os.path.join(VIDEO_DIR, LEAGUE_DIR, '2015-2016', MATCH_GAME)
VIDEO_1ST   = os.path.join(MATCH_PATH, '1_224p.mkv')
VIDEO_2ND   = os.path.join(MATCH_PATH, '2_224p.mkv')
LABELS_PATH = os.path.join(MATCH_PATH, 'Labels-v2.json')

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print()
print('=' * 55)
print('SETUP COMPLETE')
print('=' * 55)
print(f'   PROJECT_ROOT : {PROJECT_ROOT}')
print(f'   Match        : {MATCH_GAME}')
print(f'   Video 1st    : {"EXISTS" if os.path.exists(VIDEO_1ST) else "NOT FOUND"}')
print(f'   Video 2nd    : {"EXISTS" if os.path.exists(VIDEO_2ND) else "NOT FOUND"}')
print(f'   Labels       : {"EXISTS" if os.path.exists(LABELS_PATH) else "NOT FOUND"}')
print(f'   SN Password  : {SN_PASSWORD[:4]}****')
print('=' * 55)


Mounted at /content/drive
Installing dependencies...
   SoccerNet: OK
   pyocclient: OK
   statsbombpy: OK

SETUP COMPLETE
   PROJECT_ROOT : /content/drive/MyDrive/bbc-vs-msn-classifier
   Match        : 2015-11-21 - 20-15 Real Madrid 0 - 4 Barcelona
   Video 1st    : EXISTS
   Video 2nd    : EXISTS
   Labels       : NOT FOUND
   SN Password  : s0cc****


---
# ⬇️ Section 2 — Download Labels-v2.json
Downloads the annotation file for the test match. Skip if already downloaded.

In [17]:
# ============================================================
# SECTION 2 — Download Labels-v2.json
# ============================================================

if os.path.exists(LABELS_PATH):
    size_kb = os.path.getsize(LABELS_PATH) / 1024
    print(f'Labels-v2.json already exists ({size_kb:.1f} KB) — skipping download')
else:
    print(f'Downloading Labels-v2.json for: {MATCH_GAME}')
    mySNdl.downloadGame(
        game=f'{LEAGUE_DIR}/2015-2016/{MATCH_GAME}',
        files=['Labels-v2.json'],
        spl='train'
    )

# Load
with open(LABELS_PATH, 'r') as f:
    labels = json.load(f)

annotations = labels.get('annotations', [])

print(f'Labels loaded — {len(annotations)} total annotations')
print()
print(f'{"Half":<6} {"Game Time":<18} {"Label":<25} {"Team"}')
print('-' * 70)
for ann in annotations:
    half      = str(ann.get('half', '?'))
    game_time = ann.get('gameTime', '?')
    label     = ann.get('label', '?')
    team      = ann.get('team', '?')
    print(f'{half:<6} {game_time:<18} {label:<25} {team}')


Labels-v2.json already exists (41.6 KB) — skipping download
Labels loaded — 215 total annotations

Half   Game Time          Label                     Team
----------------------------------------------------------------------
?      1 - 00:00          Kick-off                  away
?      1 - 00:32          Ball out of play          not applicable
?      1 - 01:10          Throw-in                  away
?      1 - 01:22          Ball out of play          not applicable
?      1 - 01:32          Throw-in                  home
?      1 - 02:34          Ball out of play          not applicable
?      1 - 02:53          Corner                    home
?      1 - 03:15          Ball out of play          not applicable
?      1 - 03:30          Throw-in                  away
?      1 - 04:03          Foul                      away
?      1 - 04:14          Indirect free-kick        home
?      1 - 05:13          Ball out of play          not applicable
?      1 - 05:20          Throw-in     

---
# 🔍 Section 3 — Compare SoccerNet vs StatsBomb Timestamps
Extracts goals from both sources and shows them side by side.

In [18]:
# ============================================================
# SECTION 3 — Compare timestamps (CORRIGIDA)
# ============================================================

# ── SoccerNet goals
sn_goals = [
    ann for ann in annotations
    if ann.get('label', '').lower() == 'goal'
]

print(f'Goals in Labels-v2.json (SoccerNet): {len(sn_goals)}')
for g in sn_goals:
    # ✅ half está embutido no gameTime ex: '1 - 10:04'
    game_time = g.get('gameTime', '?')
    half      = game_time.split(' - ')[0] if ' - ' in game_time else '?'
    print(f"   Half {half} | {game_time:<18} | {g.get('team','?')}")

# ── StatsBomb goals
laliga = pd.read_csv(
    os.path.join(PROJECT_ROOT,
                 'data/statsbomb/events/laliga/bbc_msn_events_laliga.csv'),
    low_memory=False
)
matches = pd.read_csv(
    os.path.join(PROJECT_ROOT,
                 'data/statsbomb/matches/laliga/matches.csv')
)

def extract_team_name(value):
    if isinstance(value, dict):
        return value.get('name', str(value))
    return str(value)

matches['home'] = matches['home_team'].apply(extract_team_name)
matches['away'] = matches['away_team'].apply(extract_team_name)

clasico = matches[
    matches['home'].str.contains('Real Madrid') &
    matches['away'].str.contains('Barcelona')
].iloc[0]

match_id   = clasico['match_id']
all_events = sb.events(match_id=int(match_id))
sb_goals   = all_events[
    (all_events['type'] == 'Shot') &
    (all_events['shot_outcome'] == 'Goal')
][['player', 'timestamp', 'minute', 'second', 'period']].copy()

print(f'\nGoals in StatsBomb (match_id={match_id}): {len(sb_goals)}')
print(sb_goals.to_string(index=False))

# ── Side-by-side comparison
print()
print('=' * 70)
print('SIDE-BY-SIDE COMPARISON')
print('=' * 70)
print(f'{"#":<4} {"StatsBomb timestamp":<35} {"SoccerNet gameTime"}')
print('-' * 70)

for i, sb_row in enumerate(sb_goals.itertuples()):
    player = sb_row.player.split(' ')[-1]
    sb_str = f'P{sb_row.period} {sb_row.timestamp} ({player})'
    sn_str = sn_goals[i]['gameTime'] if i < len(sn_goals) else 'N/A'
    print(f'{i+1:<4} {sb_str:<35} {sn_str}')

Goals in Labels-v2.json (SoccerNet): 4
   Half 1 | 1 - 10:04          | away
   Half 1 | 1 - 38:35          | away
   Half 2 | 2 - 07:17          | away
   Half 2 | 2 - 28:40          | away


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(



Goals in StatsBomb (match_id=266424): 4
                       player    timestamp  minute  second  period
     Luis Alberto Suárez Díaz 00:10:06.146      10       6       1
Neymar da Silva Santos Junior 00:38:37.313      38      37       1
         Andrés Iniesta Luján 00:07:19.353      52      19       2
     Luis Alberto Suárez Díaz 00:28:42.232      73      42       2

SIDE-BY-SIDE COMPARISON
#    StatsBomb timestamp                 SoccerNet gameTime
----------------------------------------------------------------------
1    P1 00:10:06.146 (Díaz)              1 - 10:04
2    P1 00:38:37.313 (Junior)            1 - 38:35
3    P2 00:07:19.353 (Luján)             2 - 07:17
4    P2 00:28:42.232 (Díaz)              2 - 28:40


---
# 🎬 Section 4 — Extract Frames at Goal Timestamps
Extracts one frame per goal using both StatsBomb and SoccerNet timestamps.

In [19]:
# ============================================================
# SECTION 4 — Extract frames
# ============================================================

def extract_single_frame(video_path, timestamp, output_path):
    """Extract 1 frame at a given HH:MM:SS timestamp."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cmd = [
        'ffmpeg', '-y',
        '-ss', timestamp,
        '-i', video_path,
        '-vframes', '1',
        '-q:v', '2',
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return os.path.exists(output_path)


def soccernet_to_ffmpeg(game_time):
    """Converts '1 - 00:34:12' to '00:34:12'."""
    parts = game_time.split(' - ')
    return parts[1].strip() if len(parts) == 2 else game_time


print('Extracting frames...\n')
frames_info = []

for i, sb_row in enumerate(sb_goals.itertuples()):
    player = sb_row.player.split(' ')[-1]
    period = sb_row.period
    sb_ts  = sb_row.timestamp
    video  = VIDEO_1ST if period == 1 else VIDEO_2ND

    if not os.path.exists(video):
        print(f'VIDEO NOT FOUND: {video}')
        continue

    # Frame from StatsBomb timestamp
    sb_out = os.path.join(FRAMES_DIR, f'goal_{i+1}_{player}_statsbomb.jpg')
    sb_ok  = extract_single_frame(video, sb_ts, sb_out)

    # Frame from SoccerNet timestamp
    sn_ts  = None
    sn_out = None
    sn_ok  = False
    if i < len(sn_goals):
        sn_ts  = soccernet_to_ffmpeg(sn_goals[i]['gameTime'])
        sn_out = os.path.join(FRAMES_DIR, f'goal_{i+1}_{player}_soccernet.jpg')
        sn_ok  = extract_single_frame(video, sn_ts, sn_out)

    frames_info.append({
        'goal'    : i + 1,
        'player'  : player,
        'period'  : period,
        'sb_ts'   : sb_ts,
        'sn_ts'   : sn_ts,
        'sb_path' : sb_out if sb_ok else None,
        'sn_path' : sn_out if sn_ok else None,
    })

    sb_status = 'OK' if sb_ok else 'FAILED'
    sn_status = 'OK' if sn_ok else 'FAILED'
    print(f'Goal {i+1} — {player:<12} | '
          f'StatsBomb {sb_ts} [{sb_status}] | '
          f'SoccerNet {sn_ts} [{sn_status}]')

print(f'\n{len(frames_info)} goals processed')
print(f'Frames saved in: {FRAMES_DIR}')


Extracting frames...

Goal 1 — Díaz         | StatsBomb 00:10:06.146 [OK] | SoccerNet 10:04 [OK]
Goal 2 — Junior       | StatsBomb 00:38:37.313 [OK] | SoccerNet 38:35 [OK]
Goal 3 — Luján        | StatsBomb 00:07:19.353 [OK] | SoccerNet 07:17 [OK]
Goal 4 — Díaz         | StatsBomb 00:28:42.232 [OK] | SoccerNet 28:40 [OK]

4 goals processed
Frames saved in: /content/drive/MyDrive/bbc-vs-msn-classifier/data/frames/validation


---
# 👁️ Section 5 — Visual Validation + Offset Calculation
Shows StatsBomb vs SoccerNet frames side by side. Look for: ball in net, celebration, or shot moment.

In [20]:
# ============================================================
# SECTION 5 — Visual validation
# ============================================================

valid_goals = [g for g in frames_info if g['sb_path'] and g['sn_path']]

if not valid_goals:
    print('No valid frames to display.')
    print('Check that video files exist and timestamps are valid.')
else:
    fig, axes = plt.subplots(
        len(valid_goals), 2,
        figsize=(14, 5 * len(valid_goals))
    )
    fig.patch.set_facecolor('#0d1117')

    if len(valid_goals) == 1:
        axes = [axes]

    for row, goal in enumerate(valid_goals):
        for col, (key, label) in enumerate([('sb_path', 'StatsBomb'), ('sn_path', 'SoccerNet')]):
            ax  = axes[row][col]
            img = mpimg.imread(goal[key])
            ts  = goal['sb_ts'] if key == 'sb_path' else goal['sn_ts']
            ax.imshow(img)
            ax.set_title(
                f"Goal {goal['goal']} — {goal['player']}\n"
                f"{label}: {ts}",
                color='white', fontsize=10, pad=8
            )
            ax.axis('off')
            ax.set_facecolor('#0d1117')

    plt.suptitle(
        f'Sync Validation — StatsBomb (left) vs SoccerNet (right)\n{MATCH_GAME}',
        color='white', fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    save_path = os.path.join(FRAMES_DIR, 'sync_validation.png')
    plt.savefig(save_path, dpi=120, bbox_inches='tight', facecolor='#0d1117')
    print(f'Validation image saved: {save_path}')
    plt.show()

# ── Offset calculation
def ts_to_seconds(ts):
    try:
        parts = ts.split(':')
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        elif len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
    except:
        return None

print()
print('OFFSET CALCULATION (SoccerNet - StatsBomb):')
print('-' * 55)

offsets = []
for goal in frames_info:
    if goal['sn_ts'] and goal['sb_ts']:
        sb_sec = ts_to_seconds(goal['sb_ts'])
        sn_sec = ts_to_seconds(goal['sn_ts'])
        if sb_sec is not None and sn_sec is not None:
            diff = sn_sec - sb_sec
            offsets.append(diff)
            print(f"   Goal {goal['goal']} ({goal['player']:<12}): "
                  f"SB={goal['sb_ts']} | "
                  f"SN={goal['sn_ts']} | "
                  f"diff={diff:+.1f}s")

if offsets:
    print()
    print(f'   Mean offset  : {statistics.mean(offsets):+.2f}s')
    if len(offsets) > 1:
        print(f'   Stdev offset : {statistics.stdev(offsets):.2f}s')
    print(f'   Min offset   : {min(offsets):+.2f}s')
    print(f'   Max offset   : {max(offsets):+.2f}s')


Output hidden; open in https://colab.research.google.com to view.

---
# ✅ Section 6 — Sync Verdict
Automatic classification based on offset statistics.

In [21]:
# ============================================================
# SECTION 6 — Sync verdict
# ============================================================

print('=' * 60)
print('SYNC VALIDATION VERDICT')
print('=' * 60)

if not offsets:
    print('No offsets calculated.')
    print('Check Section 4 and 5 for errors.')
else:
    mean_offset  = statistics.mean(offsets)
    stdev_offset = statistics.stdev(offsets) if len(offsets) > 1 else 0.0

    print(f'\n   Goals compared : {len(offsets)}')
    print(f'   Mean offset    : {mean_offset:+.2f}s')
    print(f'   Stdev offset   : {stdev_offset:.2f}s')
    print(f'   Min offset     : {min(offsets):+.2f}s')
    print(f'   Max offset     : {max(offsets):+.2f}s')
    print()

    if abs(mean_offset) <= 2 and stdev_offset <= 1:
        verdict = 'EXCELLENT'
        msg = [
            'Timestamps are well aligned.',
            'StatsBomb timestamps can be used directly.',
            'No manual calibration needed.',
        ]
    elif abs(mean_offset) <= 5 and stdev_offset <= 2:
        verdict = 'GOOD'
        msg = [
            f'Consistent offset of ~{mean_offset:+.1f}s.',
            f'Apply a fixed offset of {mean_offset:+.1f}s to all events.',
            'Minimal manual calibration needed.',
        ]
    elif stdev_offset <= 3:
        verdict = 'ACCEPTABLE'
        msg = [
            f'Offset ~{mean_offset:+.1f}s but variable.',
            'Apply offset per match (1 anchor point per game).',
        ]
    else:
        verdict = 'POOR'
        msg = [
            'High variance in offset.',
            'Manual calibration required per match.',
        ]

    print(f'   Verdict: {verdict}')
    print()
    for line in msg:
        print(f'   {line}')

    # Save verdict
    verdict_data = {
        'match'          : MATCH_GAME,
        'goals_compared' : len(offsets),
        'mean_offset_s'  : round(mean_offset, 3),
        'stdev_offset_s' : round(stdev_offset, 3),
        'min_offset_s'   : round(min(offsets), 3),
        'max_offset_s'   : round(max(offsets), 3),
        'verdict'        : verdict,
        'offsets'        : [round(o, 3) for o in offsets],
    }
    verdict_path = os.path.join(FRAMES_DIR, 'sync_verdict.json')
    with open(verdict_path, 'w') as f:
        json.dump(verdict_data, f, indent=2)

    print()
    print(f'   Verdict saved: {verdict_path}')

print()
print('=' * 60)
print('NEXT STEPS')
print('=' * 60)
print('   EXCELLENT / GOOD  -> run extractor.py batch_extract()')
print('   ACCEPTABLE        -> apply fixed offset per match')
print('   POOR              -> manual calibration needed')
print('=' * 60)


SYNC VALIDATION VERDICT

   Goals compared : 4
   Mean offset    : -2.26s
   Stdev offset   : 0.09s
   Min offset     : -2.35s
   Max offset     : -2.15s

   Verdict: GOOD

   Consistent offset of ~-2.3s.
   Apply a fixed offset of -2.3s to all events.
   Minimal manual calibration needed.

   Verdict saved: /content/drive/MyDrive/bbc-vs-msn-classifier/data/frames/validation/sync_verdict.json

NEXT STEPS
   EXCELLENT / GOOD  -> run extractor.py batch_extract()
   ACCEPTABLE        -> apply fixed offset per match
   POOR              -> manual calibration needed


In [24]:
import subprocess, os

os.chdir(PROJECT_ROOT)
GITHUB_TOKEN = '****REMOVED****'

# --------------------------------------------------------
# Verifica o que tem no notebooks/
# --------------------------------------------------------
result = subprocess.run(['find', 'notebooks/', '-type', 'f'],
                       capture_output=True, text=True)
print('📁 Arquivos em notebooks/:\n', result.stdout)

commands = [
    # Adiciona tudo pendente
    ['git', 'add',
     'src/extractor.py',
     'requirements.txt',
     'results/comparisons/bbc_vs_msn_zonas_finalizacao.png',
     'results/pitch_plots/laliga/bbc_vs_msn_shotmap_laliga.png',
     'notebooks/'],

    ['git', 'status'],

    ['git', 'commit', '-m',
     'feat: complete sync validation pipeline and design decisions\n\n'
     '- extractor.py: 4fps, 10s window, ESRGAN 4x upscale, SYNC_OFFSET=2.26s\n'
     '- Sync validated: mean offset -2.26s, stdev 0.09s (GOOD verdict)\n'
     '- Neymar name fixed: Junior without accent (matches StatsBomb API)\n'
     '- Dataset updated: 28,119 events (6 players including Neymar)\n'
     '- Shot map and finishing zones updated with Neymar data\n'
     '- Sync validation notebook added (6 sections)\n'
     '- requirements.txt updated with pyocclient and SoccerNet\n'
     '- results: pitch plots regenerated with correct data'],

    ['git', 'push', 'origin', 'main'],
]

for cmd in commands:
    safe = ' '.join(cmd).replace(GITHUB_TOKEN, '****')
    print(f'\n$ {safe}')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)

📁 Arquivos em notebooks/:
 notebooks/SoccerNet_StatsBomb_Sync_Validation.ipynb


$ git add src/extractor.py requirements.txt results/comparisons/bbc_vs_msn_zonas_finalizacao.png results/pitch_plots/laliga/bbc_vs_msn_shotmap_laliga.png notebooks/

$ git status
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   notebooks/SoccerNet_StatsBomb_Sync_Validation.ipynb
	modified:   requirements.txt
	modified:   results/comparisons/bbc_vs_msn_zonas_finalizacao.png
	modified:   results/pitch_plots/laliga/bbc_vs_msn_shotmap_laliga.png
	modified:   src/extractor.py



$ git commit -m feat: complete sync validation pipeline and design decisions

- extractor.py: 4fps, 10s window, ESRGAN 4x upscale, SYNC_OFFSET=2.26s
- Sync validated: mean offset -2.26s, stdev 0.09s (GOOD verdict)
- Neymar name fixed: Junior without accent (matches StatsBomb API)
- Dataset updated: 28,119 events (6 players including Ne